# PySpark DataFrames without using local data 

In [25]:

from pyspark.sql import *
from pyspark.sql.functions import *

In [26]:
spark = SparkSession.builder.appName("Spark Optimization") \
.config("spark.sql.ui.explainMode", "extended").getOrCreate()



In [28]:

df = spark.read.format("csv").option("inferSchema", True).option("header", "true").option("inferSchema", "true")\
.load("BigMart Sales.csv")

In [29]:
df.rdd.getNumPartitions()

1

In [30]:
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

# Disable AQE

In [34]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [31]:
spark.conf.get("spark.sql.adaptive.enabled")

'false'

In [32]:
df_noAQE= df.groupBy("Item_fat_Content").count()
df_noAQE.show()

+----------------+-----+
|Item_fat_Content|count|
+----------------+-----+
|         low fat|  112|
|         Low Fat| 5089|
|              LF|  316|
|         Regular| 2889|
|             reg|  117|
+----------------+-----+



In [33]:
print(df_noAQE.rdd.getNumPartitions())

200


# Enable AQE

In [35]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [36]:
spark.conf.get("spark.sql.adaptive.enabled")

'true'

In [37]:
df_AQE= df.groupBy("Item_fat_Content").count()
df_AQE.show()

+----------------+-----+
|Item_fat_Content|count|
+----------------+-----+
|         low fat|  112|
|         Low Fat| 5089|
|              LF|  316|
|         Regular| 2889|
|             reg|  117|
+----------------+-----+



In [38]:
print(df_AQE.rdd.getNumPartitions())

1
